<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Proyección de la demanda externa por producto y destino

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `statsmodels`: modelos lineales generalizados y suavizamiento exponencial.
- `tqdm`: barra de avance de la proyección por producto y destino.

In [1]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.families.links import identity
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from tqdm.auto import tqdm
import unicodedata, re

/Users/juandiegoheredianino/Documents/nearshoring-colombia/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp, random_state

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

### Parámetros

Fija las ventanas temporales y los umbrales que regulan la identificación de episodios de auge
y la proyección posterior.

In [3]:
ventana_anios = 15  # Ventana de años para proyección
umbral_crec_abs = 0.06        # 6%
fraccion_acel = 1/3           # un tercio
umbral_acel_pp = 3            # 3 puntos porcentuales
anio_fin_datos = 2023
anios_proyeccion = ventana_anios          

# Grado máximo de polinomios opcional en la regresión
grado_polinomios = 1

# la semilla del proyecto vive en config.py y se importa arriba como random_state

### Lectura del panel de comercio

Carga el archivo consolidado de comercio bilateral, ajusta los tipos de las columnas y recorta
la serie hasta el último año con datos completos.

In [4]:
# Leer el archivo Parquet
df_trade = pd.read_parquet(temp / "exportaciones" / "df_comercio.parquet")


# Asegurar tipos de datos correctos
df_trade["anio"]            = df_trade["anio"].astype(int)
df_trade["hs6_1996"]        = df_trade["hs6_1996"].astype(str)
df_trade["o_iso3"]          = df_trade["o_iso3"].astype(str)
df_trade["d_iso3"]          = df_trade["d_iso3"].astype(str)
df_trade["valor_miles_usd"] = pd.to_numeric(df_trade["valor_miles_usd"], errors="coerce")

# Filtrar hasta el año tope parametrizado
df_trade = df_trade.loc[df_trade["anio"] <= anio_fin_datos].copy()

print("Observaciones totales:", len(df_trade))
df_trade.head()

Observaciones totales: 15575422


,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019
0,2004,620990,Thailand,THA,East Asia & Pacific,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,83.694,1.315
1,2004,620990,Vietnam,VNM,East Asia & Pacific,Lower middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,77.221,1.315
2,2004,621132,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,212.796,1.315
3,2004,621133,Portugal,PRT,Europe & Central Asia,High income,Angola,AGO,Sub-Saharan Africa,Lower middle income,581.756,1.315
4,2004,621142,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,67.073,1.315


### Productos priorizados

Enlaza la lista de productos con oportunidad identificada al panel comercial, para conservar el
tipo de oportunidad de cada bien.

In [5]:
# Leer la tabla de productos
df_prod = pd.read_parquet(temp / "exportaciones" / "df_productos.parquet")

# Asegurar tipos de datos correctos
df_prod["hs6_1996"] = df_prod["hs6_1996"].astype(str)

# Mantener solo las columnas necesarias
df_prod = df_prod[["hs6_1996", "tipo_oportunidad"]].copy()

# Vista rápida
print("Productos:", len(df_prod))
df_prod.head()

Productos: 809


,hs6_1996,tipo_oportunidad
0,020130,Oportunidades a profundizar
1,020312,Oportunidades a la vista
2,020319,Oportunidades a profundizar
3,020630,Oportunidades a la vista
4,021011,Oportunidades de largo alcance


### Controles y limpieza del panel

Adjunta los controles gravitacionales, imputa la similitud cultural faltante con el promedio de
la pareja de regiones y descarta las observaciones incompletas.

**Pasos realizados:**

1. **Unión de controles:** cruza por país de origen, país de destino y año.

2. **Imputación cultural:** completa el índice ausente con el promedio de la pareja de regiones correspondiente.

3. **Participaciones de mercado:** calcula el total importado de cada producto por destino y año, y la participación de cada origen en ese mercado.

In [6]:
# Leer el archivo Parquet de controles
df_ctrl = pd.read_parquet(temp / "exportaciones" / "df_controles.parquet")

list_ctrl = [x for x in df_ctrl.columns.to_list() if x not in ['o_iso3','d_iso3','anio']] + [
    #'industria',
    'tipo_oportunidad']

# Vista rápida
print("Observaciones en controles:", len(df_ctrl))

# Unir controles con el DataFrame principal
cols_key = ["o_iso3", "d_iso3", "anio"]
df_trade = df_trade.merge(df_ctrl, on=cols_key, how="left").merge(df_prod, on="hs6_1996", how="left")

print("Panel principal con controles:", df_trade.shape)
df_trade.head()

Observaciones en controles: 338065


Panel principal con controles: (15575422, 17)


,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019,food_index,tierra_cultivable_o,distancia,cultura_similaridad,tipo_oportunidad
0,2004,620990,Thailand,THA,East Asia & Pacific,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,83.694,1.315,77.543,NaN,"9,947.915",43.800,Oportunidades a profundizar
1,2004,620990,Vietnam,VNM,East Asia & Pacific,Lower middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,77.221,1.315,77.543,NaN,"10,627.528",48.600,Oportunidades a profundizar
2,2004,621132,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,212.796,1.315,77.543,7.020,"6,663.716",62.800,Oportunidades a profundizar
3,2004,621133,Portugal,PRT,Europe & Central Asia,High income,Angola,AGO,Sub-Saharan Africa,Lower middle income,581.756,1.315,77.543,NaN,"5,771.917",51.700,Oportunidades a profundizar
4,2004,621142,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,67.073,1.315,77.543,7.020,"6,663.716",62.800,Oportunidades a profundizar


In [7]:
df_inputacion_cultura = df_trade[['o_iso3','region_o','d_iso3','region_d','cultura_similaridad']].dropna().drop_duplicates().groupby(['region_o','region_d'])['cultura_similaridad'].mean().round(3).reset_index()

df_trade = df_trade.merge(df_inputacion_cultura, left_on=['region_o', 'region_d'], right_on=['region_o', 'region_d'], how='left')
df_trade['cultura_similaridad'] = df_trade['cultura_similaridad_x'].fillna(df_trade['cultura_similaridad_y'])
df_trade.drop(columns=['cultura_similaridad_x', 'cultura_similaridad_y','tierra_cultivable_o'], inplace=True)

In [8]:
# Identificar variables con al menos un valor nulo
null_vars = df_trade.columns[df_trade.isnull().any()]

for var in null_vars:
    print(f"Paises y productos donde {var} es nulo")
    subset = df_trade[df_trade[var].isnull()][['nombre_d', 'nombre_o', 'hs6_1996']]
    display(subset.drop_duplicates().reset_index(drop=True))

df_trade = df_trade.dropna()

# Construcción de shares por producto-año

# Se calcula el total que se importó del producto en cada año. 
df_trade["total_prod_year"] = (
    df_trade
      .groupby(["hs6_1996",'d_iso3', "anio"])["valor_miles_usd"]
      .transform("sum")
) # En la estimación total_prod_year sería M

# Participa del país de origen en el mercado del destino para ese producto-año.
df_trade["share"] = (
    df_trade["valor_miles_usd"] / df_trade["total_prod_year"]
) 

df_trade

Paises y productos donde distancia es nulo


,nombre_d,nombre_o,hs6_1996
0,Spain,Tokelau,300490
1,Bolivia,Tokelau,721621
2,Czech Republic,Tokelau,620342
3,Czech Republic,Tokelau,611020
4,France,Tokelau,732690
5,Czech Republic,Tokelau,610910
6,Czech Republic,Tokelau,870322
7,Czech Republic,Tokelau,870390
8,Djibouti,Tokelau,721391
9,Malaysia,Tokelau,850431


Paises y productos donde cultura_similaridad es nulo


,nombre_d,nombre_o,hs6_1996
0,Andorra,Indonesia,611020
1,Andorra,Italy,611020
2,Andorra,Malaysia,611020
3,Andorra,Netherlands,611020
4,Andorra,Thailand,611020
...,...,...,...
3526,Andorra,Switzerland,180610
3527,Andorra,Faeroe Islands,220890
3528,Andorra,United States,841989
3529,Andorra,Italy,845969


,anio,hs6_1996,nombre_o,o_iso3,region_o,ingreso_o,nombre_d,d_iso3,region_d,ingreso_d,valor_miles_usd,deflactor_2019,food_index,distancia,tipo_oportunidad,cultura_similaridad,total_prod_year,share
0,2004,620990,Thailand,THA,East Asia & Pacific,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,83.694,1.315,77.543,"9,947.915",Oportunidades a profundizar,43.800,635.111,0.132
1,2004,620990,Vietnam,VNM,East Asia & Pacific,Lower middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,77.221,1.315,77.543,"10,627.528",Oportunidades a profundizar,48.600,635.111,0.122
2,2004,621132,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,212.796,1.315,77.543,"6,663.716",Oportunidades a profundizar,62.800,212.796,1.000
3,2004,621133,Portugal,PRT,Europe & Central Asia,High income,Angola,AGO,Sub-Saharan Africa,Lower middle income,581.756,1.315,77.543,"5,771.917",Oportunidades a profundizar,51.700,581.756,1.000
4,2004,621142,Brazil,BRA,Latin America & Caribbean,Upper middle income,Angola,AGO,Sub-Saharan Africa,Lower middle income,67.073,1.315,77.543,"6,663.716",Oportunidades a profundizar,62.800,67.073,1.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15575417,1997,730890,Zimbabwe,ZWE,Sub-Saharan Africa,Lower middle income,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,87.515,1.500,82.390,398.725,Oportunidades a profundizar,82.800,"4,305.546",0.020
15575418,1997,731512,South Africa,ZAF,Sub-Saharan Africa,Upper middle income,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,118.461,1.500,82.390,"1,538.037",Oportunidades a profundizar,65.900,118.461,1.000
15575419,1997,731829,United Arab Emirates,ARE,Middle East & North Africa,High income,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,"11,900.055",1.500,82.390,"5,261.017",Oportunidades a profundizar,37.900,"12,481.796",0.953
15575420,1997,731829,Finland,FIN,Europe & Central Asia,High income,Zambia,ZMB,Sub-Saharan Africa,Lower middle income,280.453,1.500,82.390,"8,410.322",Oportunidades a profundizar,39.700,"12,481.796",0.022


### Universo de proyección

Define el conjunto de productos y de destinos sobre el que se proyecta la demanda externa.

In [9]:
productos_2023 = df_trade['hs6_1996'].unique().tolist()

destinos_colombia = [
    "ESP",  # EU - Spain
    "LCA",  # LAC - Saint Lucia
    "TTO",  # LAC - Trinidad and Tobago
    "BGR",  # EU - Bulgaria
    "COL",  # LAC - Colombia
    "USA",  # USA - United States
    "CAN",  # Canada - Canada
    "NLD",  # EU - Netherlands
    "CHL",  # LAC - Chile
    "DEU",  # EU - Germany
    "PHL",  # APEC ASIA - Philippines
    "LVA",  # EU - Latvia
    "VNM",  # APEC ASIA - Viet Nam
    "SGP",  # APEC ASIA - Singapore
    "BLZ",  # LAC - Belize
    "VEN",  # LAC - Venezuela
    "MLT",  # EU - Malta
    "CYM",  # LAC - Cayman Islands
    "FIN",  # EU - Finland
    "CRI",  # LAC - Costa Rica
    "HRV",  # EU - Croatia
    "LTU",  # EU - Lithuania
    "BRN",  # APEC ASIA - Brunei Darussalam
    "BRA",  # LAC - Brazil
    "PER",  # LAC - Peru
    "URY",  # LAC - Uruguay
    "NIC",  # LAC - Nicaragua
    "DOM",  # LAC - Dominican Republic
    "KOR",  # APEC ASIA - South Korea
    "HND",  # LAC - Honduras
    "BEL",  # EU - Belgium
    "ATG",  # LAC - Antigua and Barbuda
    "MEX",  # LAC - Mexico
    "BHS",  # LAC - Bahamas
    "HKG",  # APEC ASIA - Hong Kong
    "JAM",  # LAC - Jamaica
    "SLV",  # LAC - El Salvador
    "MYS",  # APEC ASIA - Malaysia
    "DNK",  # EU - Denmark
    "TCA",  # LAC - Turks and Caicos Islands
    "BRB",  # LAC - Barbados
    "GTM",  # LAC - Guatemala
    "CHN",  # APEC ASIA - China 
    "NZL",  # APEC ASIA - New Zealand
    "POL",  # EU - Poland
    "SWE",  # EU - Sweden
    "EST",  # EU - Estonia
    "GRC",  # EU - Greece
    "CZE",  # EU - Czechia
    "IRL",  # EU - Ireland
    "AUT",  # EU - Austria
    "LUX",  # EU - Luxembourg
    "IDN",  # APEC ASIA - Indonesia
    "AUS",  # APEC ASIA - Australia
    "JPN",  # APEC ASIA - Japan
    "SVK",  # EU - Slovakia
    "PRT",  # EU - Portugal
    "FRA",  # EU - France
    "GBR",  # UK - United Kingdom
    "ITA",  # EU - Italy
    "ARG",  # LAC - Argentina
    "PAN",  # Other - Panama  (LAC, pero ya está en lista anterior? — no aparecía; cae aquí)
    "THA",  # Other - Thailand  (APEC ASIA pero no en lista seleccionada arriba)
    "SUR",  # Other - Suriname  (LAC but no APEC ASIA; queda fuera de grupos solicitados)
    "ABW",  # Other - Aruba  (LAC territorio, pero no país soberano; fuera de grupos solicitados)
]

### Proyección de importaciones

Pronostica la demanda externa futura de cada pareja producto-destino a partir de la trayectoria
histórica de importaciones, mediante suavizamiento exponencial.

In [10]:
(temp / 'exportaciones').mkdir(parents=True, exist_ok=True)

forecast_records = []

total_iters = len(destinos_colombia) * len(productos_2023)   #  pasos totales

# EXCEPCION DE MIGRACION, POR RENDIMIENTO: el original filtraba el panel completo dentro del
# bucle, con `df_trade.loc[(df_trade["d_iso3"] == dest) & (df_trade["hs6_1996"] == prod)]`.
# Sobre 15.575.422 filas eso cuesta 801 ms por vuelta y 52.585 vueltas, es decir 11,7 horas
# solo en filtrar. Aqui se agrupa una sola vez por destino y producto, que tarda menos de un
# segundo, y el bucle consulta el grupo ya calculado. El recorrido, el orden de las filas y
# el calculo de cada serie son identicos: agrupar por las dos mismas columnas produce
# exactamente los mismos subconjuntos. Ver tools/markdown/AUDITORIA_MIGRACION.md
series_por_par = {
    clave: grupo.set_index("anio").sort_index()["total_prod_year"].dropna().drop_duplicates()
    for clave, grupo in df_trade[["d_iso3", "hs6_1996", "anio", "total_prod_year"]].groupby(
        ["d_iso3", "hs6_1996"], sort=False
    )
}

with tqdm(total=total_iters, desc="Avance global") as pbar:
    for dest in destinos_colombia:
        for prod in productos_2023:
            serie = series_por_par.get((dest, prod))

            # Si la serie tiene < 2 puntos, no podemos pronosticar → skip
            if serie is None or len(serie) < 2:
                pbar.update(1)
                continue

            # Año de la última observación
            last_obs = serie.index.max()

            # Año hasta el que queremos proyectar
            target_end = anio_fin_datos + anios_proyeccion

            # Lista dinámica de años de proyección
            anios_proj = range(last_obs + 1, target_end + 1)

            # Ajustamos ETS multiplicativo simple (sin estacionalidad)
            model_es = ExponentialSmoothing(
                serie,
                trend="add",              # suavizamiento exponencial con tendencia aditiva
                seasonal=None
            ).fit(
                optimized=False,
                smoothing_level=1,   # α bajo → menos peso al último dato
                smoothing_slope=0.1     # β bajo → tendencia más suave
            )

            # Pronóstico hasta 2030
            preds = model_es.forecast(len(anios_proj))

            for año, m_hat in zip(anios_proj, preds):
                forecast_records.append({
                    "d_iso3": dest,
                    "hs6_1996": prod,
                    "anio_pred": año,
                    "m_hat": float(m_hat)
                })
            pbar.update(1)

df_m_forecast = pd.DataFrame(forecast_records)
print("Pronósticos generados:", len(df_m_forecast))
df_m_forecast.to_parquet(temp / "exportaciones" / "df_m_forecast_6_8_25.parquet", index=False)

Avance global:   0%|          | 0/52585 [00:00<?, ?it/s]

Avance global:   0%|          | 168/52585 [00:00<00:31, 1678.84it/s]

Avance global:   1%|          | 336/52585 [00:00<00:31, 1675.63it/s]

Avance global:   1%|          | 513/52585 [00:00<00:30, 1716.28it/s]

Avance global:   1%|▏         | 685/52585 [00:00<00:30, 1709.04it/s]

Avance global:   2%|▏         | 932/52585 [00:00<00:26, 1979.79it/s]

... (214 outputs left out)

Avance global:  98%|█████████▊| 51777/52585 [00:24<00:00, 3142.64it/s]

Avance global:  99%|█████████▉| 52102/52585 [00:24<00:00, 3172.30it/s]

Avance global: 100%|██████████| 52585/52585 [00:24<00:00, 2150.40it/s]

Pronósticos generados: 718888
